# Tạo và Nạp dữ liệu cho bảng FACT_ORDERS
Bảng Fact Orders là trung tâm của Ngôi sao 1, tổng hợp từ:
- `orders_silver.csv` (thông tin đơn hàng, trạng thái)
- `payments_silver.csv` (giá trị thanh)
- `returns_silver.csv` (tổng tiền hoàn trả theo từng đơn)
Đồng thời map với các khóa ngoại từ `DIM_DATE`, `DIM_CUSTOMER`, `DIM_GEOGRAPHY`.

In [1]:
import os
from pathlib import Path

import pandas as pd
from db_engine import get_engine

engine = get_engine()

dim_customer = pd.read_sql("SELECT customer_key, customer_id FROM dbo.dim_customer", engine)
dim_geography = pd.read_sql("SELECT geography_key, zip, unit_key FROM dbo.dim_geography", engine)


In [2]:
orders = pd.read_csv(
    "../../SilverData/orders_silver.csv",
    dtype={
        "order_id": "string",
        "zip": "string",
        "customer_id": "string"
    }
)
payments = pd.read_csv(
    "../../SilverData/payments_silver.csv",
    dtype={
        "order_id": "string"
    }
)
returns = pd.read_csv(
    "../../SilverData/returns_silver.csv",
    dtype={
        "order_id": "string"
    }
)
dim_customer["customer_id"] = dim_customer["customer_id"].astype("string")
dim_geography["zip"] = dim_geography["zip"].astype("string")


refund_by_order=returns.groupby('order_id',as_index=False).agg(refund_amount=('refund_amount','sum'))
payment_by_order=payments.groupby('order_id',as_index=False).agg(payment_value=('payment_value','sum'))

df_orders=orders.merge(payment_by_order,on='order_id',how='left').merge(refund_by_order,on='order_id',how='left')

df_orders['payment_value']=df_orders['payment_value'].fillna(0).round(2)
df_orders['refund_amount']=df_orders['refund_amount'].fillna(0).round(2)

df_orders['order_date_key']=pd.to_datetime(df_orders['order_date']).dt.strftime('%Y%m%d').astype(int)

df_orders=df_orders.merge(dim_customer,on='customer_id',how='left',validate='many_to_one').merge(dim_geography,on='zip'
,how='left',validate='many_to_one')
df_orders['order_key']=df_orders.index+1
df_orders['order_id']=df_orders['order_id'].astype(str)
fact_cols=['order_key','order_id','order_date_key','customer_key','geography_key','unit_key','order_status','payment_value','refund_amount']
df_fact_orders=df_orders[fact_cols]
df_fact_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_key       646945 non-null  int64  
 1   order_id        646945 non-null  str    
 2   order_date_key  646945 non-null  int64  
 3   customer_key    646945 non-null  int64  
 4   geography_key   646945 non-null  int64  
 5   unit_key        132583 non-null  float64
 6   order_status    646945 non-null  str    
 7   payment_value   646945 non-null  float64
 8   refund_amount   646945 non-null  float64
dtypes: float64(3), int64(4), str(2)
memory usage: 44.4 MB


### Nạp dữ liệu (Load) vào Microsoft SQL Server theo Batch


In [3]:
df_fact_orders.to_sql(
    name='fact_orders',
    con=engine,
    if_exists='append',
    index=False,
    chunksize=10000
)
print(f"Đã nạp thành công {len(df_fact_orders):,} dòng vào FACT_ORDERS trên SQL Server!")


Đã nạp thành công 646,945 dòng vào FACT_ORDERS trên SQL Server!
